# 04 — Generación de respuestas con Qwen3 (Pipeline RAG completo)

**Proyecto:** Chem RAG Assistant  
**Repositorio:** https://github.com/Jesusrodriguezf90/chem-rag-assistant  
**Fase:** Prototipado del pipeline RAG completo

---

Este notebook integra todas las fases anteriores del pipeline RAG:
recuperación semántica con ChromaDB + BGE-M3 y generación de respuestas
fundamentadas con Qwen3 vía Hugging Face Inference API.

**Entrada:** base de datos ChromaDB persistida en `data/chroma_db/`  
**Salida:** respuestas en lenguaje natural fundamentadas en el documento científico,
con citación de fuentes y filtrado por umbral de similitud

In [1]:
"""
Notebook: 04_generacion.ipynb

Objetivo:
    Implementar el pipeline RAG completo integrando la recuperación
    semántica validada en 03_recuperacion.ipynb con la generación de
    respuestas mediante Qwen3 (Hugging Face Inference API).

    El pipeline aplica:
      - Filtrado por umbral mínimo de similitud antes de construir el prompt
      - Prompt específico con rol de asistente químico
      - Instrucciones de citación de fuentes
      - Comportamiento definido ante información insuficiente
      - Respuesta en el idioma de la pregunta (detección automática)

    Este notebook cubre:
      1. Configuración del entorno y credenciales
      2. Conexión a ChromaDB persistida
      3. Carga del modelo de embeddings para consultas
      4. Definición del prompt del sistema
      5. Función de recuperación con filtrado por umbral
      6. Función de generación con Qwen3
      7. Pipeline RAG completo
      8. Evaluación con las 5 preguntas de referencia
      9. Resumen de resultados

Fuente de datos:
    Büchele WRE, Schlachta TP, Gebendorfer AL, Pamperin J, Richter LF,
    Sauer MJ, Prokop A, Kühn FE. Synthesis, characterization, and biomedical
    evaluation of ethylene-bridged tetra-NHC Pd(ii), Pt(ii) and Au(iii)
    complexes, with apoptosis-inducing properties in cisplatin-resistant
    neuroblastoma cells. Frontiers in Chemistry. 2024.
    PMC: https://pmc.ncbi.nlm.nih.gov/articles/PMC10967698/

    Documento utilizado exclusivamente con fines de investigación y desarrollo.
    No se distribuye ni se incluye en el repositorio.

Autor:   Jesús Rodríguez
Fecha:   2026-04-30
Versión: 1.0.0
"""

'\nNotebook: 04_generacion.ipynb\n\nObjetivo:\n    Implementar el pipeline RAG completo integrando la recuperación\n    semántica validada en 03_recuperacion.ipynb con la generación de\n    respuestas mediante Qwen3 (Hugging Face Inference API).\n\n    El pipeline aplica:\n      - Filtrado por umbral mínimo de similitud antes de construir el prompt\n      - Prompt específico con rol de asistente químico\n      - Instrucciones de citación de fuentes\n      - Comportamiento definido ante información insuficiente\n      - Respuesta en el idioma de la pregunta (detección automática)\n\n    Este notebook cubre:\n      1. Configuración del entorno y credenciales\n      2. Conexión a ChromaDB persistida\n      3. Carga del modelo de embeddings para consultas\n      4. Definición del prompt del sistema\n      5. Función de recuperación con filtrado por umbral\n      6. Función de generación con Qwen3\n      7. Pipeline RAG completo\n      8. Evaluación con las 5 preguntas de referencia\n      

## 1. Configuración del entorno

In [2]:
# Detección del entorno de ejecución (Colab vs local)
try:
    from google.colab import drive, userdata
    IN_COLAB = True
    print("Entorno detectado: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno detectado: local")

Entorno detectado: Google Colab


In [3]:
# Montaje de Google Drive (solo en Colab)
if IN_COLAB:
    drive.mount('/content/drive')
    print("Google Drive montado correctamente")

Mounted at /content/drive
Google Drive montado correctamente


In [4]:
# Verificación del entorno de ejecución
import sys
import platform

print(f"Python : {sys.version}")
print(f"Sistema: {platform.system()} {platform.release()}")

# Detección de GPU
import torch

if torch.cuda.is_available():
    dispositivo = "cuda"
    nombre_gpu = torch.cuda.get_device_name(0)
    print(f"Dispositivo : GPU — {nombre_gpu}")
else:
    dispositivo = "cpu"
    print("Dispositivo : CPU (correcto para prototipado)")

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Sistema: Linux 6.6.113+
Dispositivo : CPU (correcto para prototipado)


## 2. Credenciales y dependencias

In [5]:
# Carga del token de Hugging Face
# En Colab: Settings → Secrets → HF_TOKEN
# En local: variable de entorno HF_TOKEN en .env
import os

if IN_COLAB:
    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')

assert HF_TOKEN, (
    "HF_TOKEN no encontrado.\n"
    "En Colab: Settings → Secrets → HF_TOKEN\n"
    "En local: añadir HF_TOKEN al archivo .env"
)

print("HF_TOKEN cargado correctamente")

HF_TOKEN cargado correctamente


In [6]:
# Instalación de dependencias
# huggingface_hub: cliente oficial para la HF Inference API
%pip install chromadb FlagEmbedding huggingface_hub -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

## 3. Definición de rutas y parámetros

In [7]:
# Librería estándar para manejo de rutas
from pathlib import Path

# Ruta raíz del proyecto en Google Drive
PROYECTO_RAIZ = Path('/content/drive/MyDrive/chem-rag-assistant')

# Ruta a ChromaDB persistida en 03_recuperacion.ipynb
DIR_CHROMA = PROYECTO_RAIZ / 'data' / 'chroma_db'

# Parámetros de recuperación
TOP_K = 4
# Umbral mínimo de similitud para incluir un chunk en el prompt
# Chunks con similitud inferior se descartan para no contaminar la respuesta
# Valor derivado del análisis de recuperación en 03_recuperacion.ipynb
UMBRAL_SIMILITUD = 0.50
NOMBRE_COLECCION = 'chem_rag_pmc10967698'

# Modelo de embeddings — debe ser idéntico al usado en la indexación
MODELO_EMBEDDINGS = 'BAAI/bge-m3'

# Modelo de generación vía HF Inference API
# Qwen3 lidera benchmarks multilingües con licencia Apache-2.0
MODELO_GENERACION = 'Qwen/Qwen3-8B'

# Límite de tokens de la respuesta generada
MAX_TOKENS_RESPUESTA = 512

# Validación de existencia de ChromaDB
assert DIR_CHROMA.exists(), (
    f"ChromaDB no encontrada: {DIR_CHROMA}\n"
    f"Ejecuta primero el notebook 03_recuperacion.ipynb"
)

print(f"ChromaDB          : {DIR_CHROMA}")
print(f"Colección         : {NOMBRE_COLECCION}")
print(f"Top-k             : {TOP_K}")
print(f"Umbral similitud  : {UMBRAL_SIMILITUD}")
print(f"Modelo embeddings : {MODELO_EMBEDDINGS}")
print(f"Modelo generación : {MODELO_GENERACION}")
print(f"Max tokens        : {MAX_TOKENS_RESPUESTA}")

ChromaDB          : /content/drive/MyDrive/chem-rag-assistant/data/chroma_db
Colección         : chem_rag_pmc10967698
Top-k             : 4
Umbral similitud  : 0.5
Modelo embeddings : BAAI/bge-m3
Modelo generación : Qwen/Qwen3-8B
Max tokens        : 512


## 4. Conexión a ChromaDB y modelo de embeddings

In [8]:
# Conexión a la base de datos vectorial persistida
import chromadb

cliente  = chromadb.PersistentClient(path=str(DIR_CHROMA))
coleccion = cliente.get_collection(name=NOMBRE_COLECCION)

print(f"Colección cargada  : {NOMBRE_COLECCION}")
print(f"Documentos en índice: {coleccion.count()}")

Colección cargada  : chem_rag_pmc10967698
Documentos en índice: 77


In [9]:
# Carga del modelo de embeddings para consultas
# Debe ser idéntico al modelo usado en la indexación
from FlagEmbedding import BGEM3FlagModel

print(f"Cargando modelo {MODELO_EMBEDDINGS}...")

modelo_embeddings = BGEM3FlagModel(
    MODELO_EMBEDDINGS,
    use_fp16=True,
    device=dispositivo,
)

print(f"Modelo cargado en  : {dispositivo.upper()}")

Cargando modelo BAAI/bge-m3...


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Modelo cargado en  : CPU


## 5. Definición del prompt del sistema

In [10]:
# Prompt del sistema con rol específico de asistente químico
# Define: rol, comportamiento ante información insuficiente,
# formato de respuesta, instrucciones de citación e idioma
SYSTEM_PROMPT = """\
You are a specialized scientific assistant in organometallic chemistry \
and medicinal inorganic chemistry. Your role is to answer questions \
accurately and concisely based exclusively on the provided document context.

<guidelines>
- Answer ONLY based on the information present in the provided context.
- If the context does not contain enough information to answer the \
question, explicitly state: "The provided document does not contain \
sufficient information to answer this question."
- Do NOT speculate or introduce information not present in the context.
- First cite the relevant fragment that supports your answer, \
then provide your conclusion.
- Use precise scientific terminology consistent with the document.
- Respond in the same language as the user's question.
- Keep your answer focused and concise (maximum 3-4 paragraphs).
</guidelines>

<example>
Question: What ligand type is used in the complexes?
Context fragment: "N-heterocyclic carbenes (NHCs) are used as ligands \
in the synthesis of Pd(II) and Pt(II) complexes."
Answer: According to the document, the complexes use \
N-heterocyclic carbenes (NHCs) as ligands, specifically in \
the synthesis of Pd(II) and Pt(II) complexes.
</example>
"""

print("System prompt definido correctamente")
print(f"Longitud del prompt : {len(SYSTEM_PROMPT)} caracteres")

System prompt definido correctamente
Longitud del prompt : 1204 caracteres


## 6. Función de recuperación con filtrado por umbral

In [11]:
# Función de recuperación semántica con filtrado por umbral mínimo
# Los chunks con similitud inferior al umbral se descartan antes
# de construir el prompt para evitar contaminar la respuesta
# con contexto irrelevante
#
# NOTA PARA MIGRACIÓN A src/retrieval/vector_store.py:
# 1. Añadir try/except con logging para trazabilidad en producción
# 2. Añadir filtrado por metadatos (fuente=) para soporte multi-documento
def recuperar_chunks_filtrados(
    pregunta: str,
    top_k: int = TOP_K,
    umbral: float = UMBRAL_SIMILITUD,
) -> tuple[list[str], list[float]]:
    """Recupera y filtra chunks por umbral mínimo de similitud.

    Args:
        pregunta: texto de la consulta del usuario
        top_k: número máximo de chunks a recuperar
        umbral: similitud mínima para incluir un chunk en el contexto

    Returns:
        Tupla (chunks_filtrados, similitudes_filtradas)
    """
    # Generar embedding de la consulta
    resultado = modelo_embeddings.encode(
        [pregunta],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=False,
        return_colbert_vecs=False,
    )
    vector_consulta = resultado['dense_vecs'][0].tolist()

    # Búsqueda por similitud en ChromaDB
    resultados = coleccion.query(
        query_embeddings=[vector_consulta],
        n_results=top_k,
        include=["documents", "distances"],
    )

    chunks     = resultados['documents'][0]
    distancias = resultados['distances'][0]
    # ChromaDB con métrica coseno devuelve distancias (1 - similitud)
    similitudes = [round(1 - d, 4) for d in distancias]

    # Filtrar chunks descartando los que no superan el umbral mínimo
    # Este paso no existe en 03_recuperacion — es exclusivo del pipeline
    # de generación para garantizar que el LLM solo recibe contexto relevante
    chunks_filtrados = [
        chunk for chunk, sim in zip(chunks, similitudes)
        if sim >= umbral
    ]
    similitudes_filtradas = [
        sim for sim in similitudes
        if sim >= umbral
    ]

    return chunks_filtrados, similitudes_filtradas


print("Función de recuperación con filtrado definida correctamente")

Función de recuperación con filtrado definida correctamente


## 7. Función de generación con Qwen3

In [17]:
# Función de generación de respuestas con Qwen3
# vía Hugging Face Inference API (serverless, gratuita)
from huggingface_hub import InferenceClient
import re

# Inicialización del cliente de inferencia
cliente_llm = InferenceClient(
    model=MODELO_GENERACION,
    token=HF_TOKEN,
)


def generar_respuesta(
    pregunta: str,
    chunks_contexto: list[str],
    similitudes: list[float],
) -> str:
    """Genera una respuesta fundamentada en el contexto recuperado.

    Args:
        pregunta: texto de la consulta del usuario
        chunks_contexto: fragmentos del documento recuperados y filtrados
        similitudes: puntuaciones de similitud de cada chunk

    Returns:
        Respuesta generada por el LLM en el idioma de la pregunta
    """
    # Si no hay chunks que superen el umbral informar al usuario
    if not chunks_contexto:
        return (
            "The provided document does not contain sufficient information "
            "to answer this question with the required confidence level."
        )

    # Construir el contexto numerando cada chunk con su similitud
    contexto_formateado = "\n\n".join([
        f"[Fragmento {i + 1} | similitud: {sim:.4f}]\n{chunk}"
        for i, (chunk, sim) in enumerate(zip(chunks_contexto, similitudes))
    ])

    # Construir el mensaje de usuario con contexto y pregunta
    # /no_think suprime el modo de razonamiento interno de Qwen3
    # evitando el bloque <think>...</think> en la respuesta
    mensaje_usuario = (
        f"Context from the scientific document:\n\n"
        f"{contexto_formateado}\n\n"
        f"Question: {pregunta}\n\n"
        f"/no_think"
    )

    # Llamada a la HF Inference API con Qwen3
    respuesta = cliente_llm.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": mensaje_usuario},
        ],
        max_tokens=MAX_TOKENS_RESPUESTA,
        temperature=0.1,
    )

    contenido = respuesta.choices[0].message.content

    # Filtrado defensivo del bloque <think>...</think>
    # como salvaguarda en caso de que /no_think no sea respetado por la API
    contenido_limpio = re.sub(
        r'<think>.*?</think>\s*',
        '',
        contenido,
        flags=re.DOTALL,
    ).strip()

    return contenido_limpio


print("Función de generación definida correctamente")

Función de generación definida correctamente


## 8. Pipeline RAG completo

In [18]:
# Pipeline RAG completo: recuperación + filtrado + generación
# Integra todas las fases del proyecto en una única función
# El tiempo de ejecución se registra para monitorización y detección
# de cuellos de botella al migrar a la API FastAPI
import time

def pipeline_rag(pregunta: str) -> dict:
    """Ejecuta el pipeline RAG completo para una pregunta dada.

    Args:
        pregunta: consulta del usuario en cualquier idioma

    Returns:
        Diccionario con la respuesta, chunks usados, similitudes
        y tiempo total de ejecución en segundos
    """
    inicio = time.time()

    # Fase 1: Recuperación y filtrado
    chunks_filtrados, similitudes = recuperar_chunks_filtrados(pregunta)

    # Fase 2: Generación
    respuesta = generar_respuesta(pregunta, chunks_filtrados, similitudes)

    return {
        "pregunta"       : pregunta,
        "respuesta"      : respuesta,
        "chunks"         : chunks_filtrados,
        "similitudes"    : similitudes,
        "n_chunks"       : len(chunks_filtrados),
        "tiempo_total_s" : round(time.time() - inicio, 2),
    }


print("Pipeline RAG completo definido correctamente")

Pipeline RAG completo definido correctamente


## 9. Evaluación con las 5 preguntas de referencia

In [19]:
# Preguntas de evaluación con ground truth
# Definidas en 03_recuperacion.ipynb y reutilizadas aquí
# para comparar las respuestas generadas contra la referencia
PREGUNTAS_EVALUACION = [
    {
        "pregunta"    : "What metals are used in the NHC complexes studied?",
        "nivel"       : "básica",
        "ground_truth": "Palladium (Pd), platinum (Pt) and gold (Au).",
    },
    {
        "pregunta"    : "What is the effect of the complexes on cisplatin-resistant neuroblastoma cells?",
        "nivel"       : "básica",
        "ground_truth": (
            "The complexes induce apoptosis in cisplatin-resistant "
            "SK-N-AS neuroblastoma cells."
        ),
    },
    {
        "pregunta"    : "What analytical techniques were used to characterize the compounds?",
        "nivel"       : "básica",
        "ground_truth": (
            "NMR spectroscopy (1H, 13C, 31P), ESI mass spectrometry "
            "and IR spectroscopy."
        ),
    },
    {
        "pregunta"    : "What is the role of the ethylene bridge in the tetra-NHC ligand design?",
        "nivel"       : "intermedia",
        "ground_truth": (
            "The ethylene bridge connects two NHC units forming a tetradentate "
            "chelating ligand that stabilizes the metal center."
        ),
    },
    {
        "pregunta"    : "How do the cytotoxicity results of the Au(III) complexes compare to cisplatin?",
        "nivel"       : "avanzada",
        "ground_truth": (
            "The Au(III) complexes show higher cytotoxicity than cisplatin "
            "in resistant cell lines, overcoming cisplatin resistance."
        ),
    },
]

print(f"Preguntas de evaluación cargadas: {len(PREGUNTAS_EVALUACION)}")

Preguntas de evaluación cargadas: 5


In [20]:
# Ejecución del pipeline RAG completo sobre las 5 preguntas
print("=" * 60)
print("EVALUACIÓN — PIPELINE RAG COMPLETO")
print(f"Modelo recuperación : {MODELO_EMBEDDINGS}")
print(f"Modelo generación   : {MODELO_GENERACION}")
print(f"Top-k               : {TOP_K}")
print(f"Umbral similitud    : {UMBRAL_SIMILITUD}")
print("=" * 60)

resultados_pipeline = []

for item in PREGUNTAS_EVALUACION:
    pregunta     = item['pregunta']
    nivel        = item['nivel']
    ground_truth = item['ground_truth']

    print(f"\n[{nivel.upper()}] {pregunta}")
    print(f"Ground truth: {ground_truth}")
    print("-" * 60)

    resultado = pipeline_rag(pregunta)

    print(f"Chunks usados   : {resultado['n_chunks']} de {TOP_K}")
    print(f"Tiempo total    : {resultado['tiempo_total_s']} s")
    if resultado['similitudes']:
        print(
            f"Similitud top-1 : {resultado['similitudes'][0]:.4f}"
        )
    print()
    print("RESPUESTA GENERADA:")
    print(resultado['respuesta'])
    print()

    resultados_pipeline.append({
        "pregunta"      : pregunta,
        "nivel"         : nivel,
        "ground_truth"  : ground_truth,
        "respuesta"     : resultado['respuesta'],
        "n_chunks"      : resultado['n_chunks'],
        "similitudes"   : resultado['similitudes'],
        "tiempo_total_s": resultado['tiempo_total_s'],
    })

EVALUACIÓN — PIPELINE RAG COMPLETO
Modelo recuperación : BAAI/bge-m3
Modelo generación   : Qwen/Qwen3-8B
Top-k               : 4
Umbral similitud    : 0.5

[BÁSICA] What metals are used in the NHC complexes studied?
Ground truth: Palladium (Pd), platinum (Pt) and gold (Au).
------------------------------------------------------------


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  3.57it/s]


Chunks usados   : 4 de 4
Tiempo total    : 1.89 s
Similitud top-1 : 0.6227

RESPUESTA GENERADA:
According to the document, the metals used in the NHC complexes studied include palladium (Pd), platinum (Pt), and gold (Au). The synthesis and characterization of complexes such as Pd/PtL3, PdL5/6, Pd/PtL8, and Pd/AuL9 are mentioned, indicating the use of these metals in the complexes. Additionally, the document refers to the application of these complexes in medicinal chemistry and catalysis.


[BÁSICA] What is the effect of the complexes on cisplatin-resistant neuroblastoma cells?
Ground truth: The complexes induce apoptosis in cisplatin-resistant SK-N-AS neuroblastoma cells.
------------------------------------------------------------


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.68it/s]


Chunks usados   : 4 de 4
Tiempo total    : 1.99 s
Similitud top-1 : 0.5773

RESPUESTA GENERADA:
According to the document, the complexes, specifically AuL9, are effective in inducing apoptosis in cisplatin-resistant neuroblastoma cells. The data suggest that AuL9 is capable of overcoming resistance to cisplatin in neuroblastoma cells (SK-N-AS) in vitro. Furthermore, it is shown that AuL9 was also effective in inducing apoptosis in cisplatin-resistant cells, thus overcoming resistance. The effect was demonstrated through experiments where nuclear DNA fragmentation was analyzed by flow cytometric analysis.


[BÁSICA] What analytical techniques were used to characterize the compounds?
Ground truth: NMR spectroscopy (1H, 13C, 31P), ESI mass spectrometry and IR spectroscopy.
------------------------------------------------------------


...

Chunks usados   : 4 de 4
Tiempo total    : 2.06 s
Similitud top-1 : 0.5374

RESPUESTA GENERADA:
The document mentions the use of NMR spectroscopy for the characterization of compounds. Specifically, it refers to the observation of complex signals in the aliphatic region of the 1 H-NMR spectrum after purification, indicating the decomposition of H2L3. Additionally, the document mentions the use of NMR to determine the purity of the kinetically preferred C[4] unit in H4L5 and H4L8, with the purity increased to 98% C[4] for H4L5 and 100% for H4L8. 

The provided document does not explicitly mention other analytical techniques such as mass spectrometry, X-ray crystallography, or infrared spectroscopy. Therefore, based on the information given, the primary analytical technique used for characterizing the compounds is nuclear magnetic resonance (NMR) spectroscopy.


[INTERMEDIA] What is the role of the ethylene bridge in the tetra-NHC ligand design?
Ground truth: The ethylene bridge connects

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


Chunks usados   : 4 de 4
Tiempo total    : 3.01 s
Similitud top-1 : 0.5667

RESPUESTA GENERADA:
The ethylene bridge plays a crucial role in the design of the tetra-NHC ligands by enabling the formation of multidentate ligands. According to the document, the study extends the scope of multidentate NHC ligands using an ethylene-bridged bisimidazolinium ligand precursor and two cyclic ethylene-bridged tetradentate NHC ligands, with an unsaturated (imidazole) and saturated backbone (2-imidazoline). This structural feature allows for the synthesis of Pd(II), Pt(II), and Au(III) tetracarbene complexes, which are characterized and evaluated for their biomedical properties.

The ethylene bridge contributes to the structural diversity and functionality of the ligands, facilitating the formation of stable complexes with transition metals. The document highlights that the ethylene-bridged ligands are essential for the synthesis of the tetracarbene complexes, which are then used in preliminary med

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


Chunks usados   : 4 de 4
Tiempo total    : 2.12 s
Similitud top-1 : 0.5555

RESPUESTA GENERADA:
According to the document, AuL9 was effective in inducing apoptosis in cisplatin-resistant neuroblastoma cells, thus overcoming resistance. The document states that 8.25 m Mcisplatin has been used as a positive control to prove resistance, indicating that AuL9 demonstrates cytotoxic activity comparable to cisplatin in overcoming resistance. However, the document also notes that a relatively high dose of AuL9 is required to induce apoptosis in neuroblastoma cells, which could be challenging for clinical applicability. This suggests that while AuL9 shows promise in overcoming cisplatin resistance, its cytotoxicity may not be as potent or efficient as cisplatin in certain contexts.



## 10. Resumen de resultados

In [21]:
# Resumen cuantitativo del pipeline RAG completo
print("=" * 60)
print("RESUMEN — PIPELINE RAG COMPLETO")
print("=" * 60)
print(f"  Modelo recuperación : {MODELO_EMBEDDINGS}")
print(f"  Modelo generación   : {MODELO_GENERACION}")
print(f"  Umbral similitud    : {UMBRAL_SIMILITUD}")
print(f"  Preguntas evaluadas : {len(resultados_pipeline)}")
print()
print(
    f"  {'Nivel':<12} {'Chunks':>6} {'Sim. top-1':>10} "
    f"{'Tiempo (s)':>10} {'Pregunta':<30}"
)
print(f"  {'-' * 70}")

for r in resultados_pipeline:
    sim_top1 = r['similitudes'][0] if r['similitudes'] else 0.0
    print(
        f"  {r['nivel']:<12} {r['n_chunks']:>6} "
        f"{sim_top1:>10.4f} {r['tiempo_total_s']:>10.2f} "
        f"{r['pregunta'][:30]}"
    )

tiempo_medio = round(
    sum(r['tiempo_total_s'] for r in resultados_pipeline)
    / len(resultados_pipeline), 2
)
print(f"  {'-' * 70}")
print(f"  Tiempo medio por pregunta : {tiempo_medio} s")
print("=" * 60)
print("Pipeline RAG completo validado")
print("Siguiente paso: 02b_chunking_eval.ipynb")

RESUMEN — PIPELINE RAG COMPLETO
  Modelo recuperación : BAAI/bge-m3
  Modelo generación   : Qwen/Qwen3-8B
  Umbral similitud    : 0.5
  Preguntas evaluadas : 5

  Nivel        Chunks Sim. top-1 Tiempo (s) Pregunta                      
  ----------------------------------------------------------------------
  básica            4     0.6227       1.89 What metals are used in the NH
  básica            4     0.5773       1.99 What is the effect of the comp
  básica            4     0.5374       2.06 What analytical techniques wer
  intermedia        4     0.5667       3.01 What is the role of the ethyle
  avanzada          4     0.5555       2.12 How do the cytotoxicity result
  ----------------------------------------------------------------------
  Tiempo medio por pregunta : 2.21 s
Pipeline RAG completo validado
Siguiente paso: 02b_chunking_eval.ipynb
